# Homework 6 - Arboles de decision y ensambles
ML Zoomcamp 2026 - Laura Velez

objetivo: predecir `fuel_efficiency_mpg` (regresion) con arboles, random forest y xgboost

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

### datos

In [2]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
df = pd.read_csv(data)
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [3]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

el enunciado dice rellenar todo con 0 (esta vez tambien las de texto, no 'NA')

In [4]:
df = df.fillna(0)
df.isnull().sum().sum()

np.int64(0)

### split
exacto como en el enunciado

In [5]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

len(df_train), len(df_val), len(df_test)

(6000, 2000, 2000)

In [6]:
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

### DictVectorizer
sparse=True como pide la tarea

In [7]:
dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))

X_train.shape

(6000, 16)

### Q1 - arbol con max_depth=1

In [8]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



con max_depth=1 el arbol solo hace un corte. el nombre que sale ahi es la variable (si sale algo como `origin=USA`, la variable es `origin`)

### Q2 - random forest con 10 arboles

In [9]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
root_mean_squared_error(y_val, y_pred)

1.8332197358745623

### Q3 - probar n_estimators

In [10]:
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    rmse = root_mean_squared_error(y_val, rf.predict(X_val))
    print(n, round(rmse, 3))

10 1.833
50 1.774
100 1.77
150 1.769


### Q4 - mejor max_depth (promedio del rmse sobre los n_estimators)
esta celda se demora un poco, son 16 modelos

In [11]:
resultados = []

for d in [10, 15, 20, 25]:
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        rmse = root_mean_squared_error(y_val, rf.predict(X_val))
        resultados.append((d, n, rmse))

df_res = pd.DataFrame(resultados, columns=['max_depth', 'n_estimators', 'rmse'])
df_res.groupby('max_depth').rmse.mean()

max_depth
10    1.754392
15    1.784224
20    1.786771
25    1.786305
Name: rmse, dtype: float64

### Q5 - importancia de variables

In [12]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

importancias = pd.Series(rf.feature_importances_, index=dv.get_feature_names_out())
importancias.sort_values(ascending=False)

model_year                      0.296107
vehicle_weight                  0.203313
fuel_type=Gasoline              0.160712
horsepower                      0.078271
engine_displacement             0.061281
acceleration                    0.054998
origin=USA                      0.042152
drivetrain=Front-wheel drive    0.020062
num_doors                       0.019401
fuel_type=Diesel                0.016571
fuel_type=Hybrid                0.016272
drivetrain=All-wheel drive      0.007951
num_cylinders                   0.007318
origin=Asia                     0.006515
drivetrain=Rear-wheel drive     0.005638
origin=Europe                   0.003436
dtype: float64

In [13]:
# solo las 4 que pregunta la tarea
importancias[['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']].sort_values(ascending=False)

vehicle_weight         0.203313
horsepower             0.078271
engine_displacement    0.061281
acceleration           0.054998
dtype: float64

### Q6 - xgboost
si no esta instalado: `%pip install xgboost`

In [14]:
import xgboost as xgb

features = list(dv.get_feature_names_out())

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

watchlist = [(dtrain, 'train'), (dval, 'val')]

funcion para sacar el rmse final de validacion de cada modelo

In [15]:
def entrenar_xgb(eta):
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,

        'objective': 'reg:squarederror',
        'nthread': 8,

        'seed': 1,
        'verbosity': 1,
    }
    evals = {}
    model = xgb.train(xgb_params, dtrain, num_boost_round=100,
                      evals=watchlist, evals_result=evals, verbose_eval=False)
    return evals['val']['rmse'][-1]

In [16]:
for eta in [0.3, 0.1]:
    print(eta, round(entrenar_xgb(eta), 4))

0.3 1.8265
0.1 1.7248
